# Mosquito Suitability: Dashboard Insights

**Purpose:** Transform exploratory data into guided analysis with quantified findings.

## Outputs
| File | Use in Tableau |
|------|----------------|
| `summary_statistics.csv` | KPI boxes, BANs |
| `insight_annotations.csv` | Dashboard text annotations |
| `city_insights_extended.csv` | Enhanced flags for filtering & highlighting |

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Config
INPUT_FILE = 'mosquito_suitability.csv'
THRESHOLD = 0.3

## 1. Load & Prepare Data

In [2]:
# Load raw data
df_long = pd.read_csv(INPUT_FILE)
df_long['city_key'] = df_long['city'] + ', ' + df_long['country']
print(f"Loaded {df_long['city_key'].nunique():,} cities")

# Pivot to wide
id_cols = ['city_key', 'city', 'country', 'lat', 'lon', 'population', 'elevation_m']
city_ids = df_long[id_cols + ['month']].drop_duplicates(subset=['city_key']).drop(columns=['month'])

aegypti_wide = df_long.pivot(index='city_key', columns='month', values='suitability_score_aegypti').add_prefix('aegypti_')
albopictus_wide = df_long.pivot(index='city_key', columns='month', values='suitability_score_albopictus').add_prefix('albopictus_')
vpd_wide = df_long.pivot(index='city_key', columns='month', values='vpd_kpa').add_prefix('vpd_')

df = city_ids.set_index('city_key').join([aegypti_wide, albopictus_wide, vpd_wide]).reset_index()

Loaded 1,423 cities


In [3]:
# Helper functions
def get_monthly(row, prefix):
    return [row[f'{prefix}_{m}'] for m in range(1, 13)]

def season_length(values, threshold):
    return sum(1 for v in values if v >= threshold)

def season_start(values, threshold):
    for i, v in enumerate(values):
        if v >= threshold: return i + 1
    return np.nan

def season_end(values, threshold):
    last = np.nan
    for i, v in enumerate(values):
        if v >= threshold: last = i + 1
    return last

def calc_peak(values):
    return max(values)

def calc_cumulative(values):
    return sum(values)

def bimodal_flag(values, threshold, vpd_values=None):
    june, july, august = values[5], values[6], values[7]
    season_len = sum(1 for v in values if v >= threshold)

    # Original: symmetric mid-summer dip (e.g. humid subtropical)
    has_symmetric_dip = (july < june) and (july < august)
    shoulders_above = (june >= threshold) and (august >= threshold)
    symmetric_bimodal = has_symmetric_dip and shoulders_above and season_len >= 4

    # Extended: asymmetric VPD-driven suppression (e.g. Mediterranean)
    # June is peak, suitability declines through summer due to heat/drought
    if vpd_values is not None:
        vpd_july = vpd_values[6]
        has_june_peak = (june > july) and (june >= threshold)
        vpd_suppressed = vpd_july > 1.2
        asymmetric_med = has_june_peak and vpd_suppressed and season_len >= 3
    else:
        asymmetric_med = False

    return symmetric_bimodal or asymmetric_med

def summer_vpd(vpd_values, lat):
    if lat > 0:
        return np.mean([vpd_values[5], vpd_values[6], vpd_values[7]])
    else:
        return np.mean([vpd_values[11], vpd_values[0], vpd_values[1]])

def lat_zone(lat):
    abs_lat = abs(lat)
    if abs_lat <= 23.5: return 'Tropics'
    elif abs_lat <= 40: return 'Subtropics'
    else: return 'Temperate'

def suitability_tier(season_len, peak):
    if season_len >= 9 and peak >= 0.7: return 'Extended'
    elif season_len >= 4 and peak >= 0.4: return 'Seasonal'
    elif season_len >= 1: return 'Marginal'
    else: return 'Minimal'

In [4]:
# Calculate base features
city = df[['city_key', 'city', 'country', 'lat', 'lon', 'population', 'elevation_m']].copy()

aeg_vals = df.apply(lambda r: get_monthly(r, 'aegypti'), axis=1)
alb_vals = df.apply(lambda r: get_monthly(r, 'albopictus'), axis=1)
vpd_vals = df.apply(lambda r: get_monthly(r, 'vpd'), axis=1)

city['season_aegypti']          = aeg_vals.apply(lambda v: season_length(v, THRESHOLD))
city['season_albopictus']       = alb_vals.apply(lambda v: season_length(v, THRESHOLD))
city['season_start_aegypti']    = aeg_vals.apply(lambda v: season_start(v, THRESHOLD))
city['season_start_albopictus'] = alb_vals.apply(lambda v: season_start(v, THRESHOLD))
city['season_end_aegypti']      = aeg_vals.apply(lambda v: season_end(v, THRESHOLD))
city['season_end_albopictus']   = alb_vals.apply(lambda v: season_end(v, THRESHOLD))
city['peak_score_aegypti']      = aeg_vals.apply(calc_peak)
city['peak_score_albopictus']   = alb_vals.apply(calc_peak)
city['cumulative_aegypti']      = aeg_vals.apply(calc_cumulative)
city['cumulative_albopictus']   = alb_vals.apply(calc_cumulative)
city['bimodal_flag_aegypti']    = pd.Series([bimodal_flag(a, THRESHOLD, v) for a, v in zip(aeg_vals, vpd_vals)])
city['bimodal_flag_albopictus'] = pd.Series([bimodal_flag(a, THRESHOLD, v) for a, v in zip(alb_vals, vpd_vals)])
city['summer_vpd']              = df.apply(lambda r: summer_vpd(get_monthly(r, 'vpd'), r['lat']), axis=1)
city['lat_zone']                = df['lat'].apply(lat_zone)
city['abs_lat']                 = df['lat'].abs()

city['altitude_suppressed'] = (
    (city['lat'].abs() <= 23.5) &
    (city['elevation_m'] > 1500) &
    (city['season_aegypti'] == 0)
)

city['species_divergence'] = city['season_albopictus'] - city['season_aegypti']

# Peak month (month with highest suitability score)
city['peak_month_aegypti']    = aeg_vals.apply(lambda v: v.index(max(v)) + 1)
city['peak_month_albopictus'] = alb_vals.apply(lambda v: v.index(max(v)) + 1)

city['suitability_tier_aegypti'] = city.apply(
    lambda r: suitability_tier(r['season_aegypti'], r['peak_score_aegypti']), axis=1
)
city['suitability_tier_albopictus'] = city.apply(
    lambda r: suitability_tier(r['season_albopictus'], r['peak_score_albopictus']), axis=1
)

print(f"✓ Base features calculated for {len(city):,} cities")

✓ Base features calculated for 1,423 cities


## 2. Generate Summary Statistics (A)

In [5]:
# Calculate all summary statistics
summary = {}

# Basic counts
n_cities = len(city)
summary['total_cities'] = n_cities

# Suitability tiers
summary['extended_suitability_aegypti'] = (city['suitability_tier_aegypti'] == 'Extended').sum()
summary['extended_suitability_albopictus'] = (city['suitability_tier_albopictus'] == 'Extended').sum()
summary['extended_suitability_aegypti_pct'] = round(summary['extended_suitability_aegypti'] / n_cities * 100, 1)
summary['extended_suitability_albopictus_pct'] = round(summary['extended_suitability_albopictus'] / n_cities * 100, 1)

# Bimodal pattern
summary['bimodal_aegypti'] = city['bimodal_flag_aegypti'].sum()
summary['bimodal_albopictus'] = city['bimodal_flag_albopictus'].sum()
summary['bimodal_albopictus_pct'] = round(summary['bimodal_albopictus'] / n_cities * 100, 1)

# Bimodal by zone
bimodal_cities = city[city['bimodal_flag_albopictus']]
bimodal_non_tropical = bimodal_cities[bimodal_cities['lat_zone'] != 'Tropics']
summary['bimodal_non_tropical_pct'] = round(len(bimodal_non_tropical) / len(bimodal_cities) * 100, 1) if len(bimodal_cities) > 0 else 0

# Altitude protection
summary['altitude_protected'] = city['altitude_suppressed'].sum()
tropical_high = city[(city['lat'].abs() <= 23.5) & (city['elevation_m'] > 1500)]
summary['altitude_protected_pct'] = round(summary['altitude_protected'] / len(tropical_high) * 100, 1) if len(tropical_high) > 0 else 0

# Species divergence by zone
for zone in ['Tropics', 'Subtropics', 'Temperate']:
    zone_data = city[city['lat_zone'] == zone]
    summary[f'median_season_aegypti_{zone.lower()}'] = zone_data['season_aegypti'].median()
    summary[f'median_season_albopictus_{zone.lower()}'] = zone_data['season_albopictus'].median()
    summary[f'median_divergence_{zone.lower()}'] = zone_data['species_divergence'].median()
    summary[f'city_count_{zone.lower()}'] = len(zone_data)

# Latitude gradient regression (full dataset — used for KPI)
slope_aeg, intercept_aeg, r_aeg, p_aeg, _ = stats.linregress(city['abs_lat'], city['season_aegypti'])
slope_alb, intercept_alb, r_alb, p_alb, _ = stats.linregress(city['abs_lat'], city['season_albopictus'])
summary['lat_gradient_aegypti_per_10deg'] = round(slope_aeg * 10, 2)
summary['lat_gradient_albopictus_per_10deg'] = round(slope_alb * 10, 2)
summary['lat_gradient_aegypti_r2'] = round(r_aeg**2, 3)
summary['lat_gradient_albopictus_r2'] = round(r_alb**2, 3)

# Post-tropics regression (>23.5° only) — documents species divergence
# Note: albopictus R²=0.10 reflects plateau behaviour above 45°, not poor fit
post_tropics = city[city['abs_lat'] > 23.5]
slope_aeg_pt, _, r_aeg_pt, _, _ = stats.linregress(post_tropics['abs_lat'], post_tropics['season_aegypti'])
slope_alb_pt, _, r_alb_pt, _, _ = stats.linregress(post_tropics['abs_lat'], post_tropics['season_albopictus'])
summary['lat_gradient_aegypti_per_10deg_posttropics'] = round(slope_aeg_pt * 10, 2)
summary['lat_gradient_albopictus_per_10deg_posttropics'] = round(slope_alb_pt * 10, 2)
summary['lat_gradient_aegypti_r2_posttropics'] = round(r_aeg_pt**2, 3)
summary['lat_gradient_albopictus_r2_posttropics'] = round(r_alb_pt**2, 3)

# Extended suitability distribution by zone
extended_aeg = city[city['suitability_tier_aegypti'] == 'Extended']
summary['extended_suitability_tropical_pct'] = round((extended_aeg['lat_zone'] == 'Tropics').sum() / len(extended_aeg) * 100, 1) if len(extended_aeg) > 0 else 0
summary['extended_suitability_subtropical_pct'] = round((extended_aeg['lat_zone'] == 'Subtropics').sum() / len(extended_aeg) * 100, 1) if len(extended_aeg) > 0 else 0

# Season timing
temperate = city[city['lat_zone'] == 'Temperate']
summary['temperate_median_start_albopictus'] = temperate['season_start_albopictus'].median()
summary['temperate_median_end_albopictus'] = temperate['season_end_albopictus'].median()

# Hemisphere distribution and peak month stats
nh_cities = city[city['lat'] > 0]
sh_cities = city[city['lat'] < 0]
summary['nh_city_count'] = len(nh_cities)
summary['sh_city_count'] = len(sh_cities)
summary['nh_city_pct'] = round(len(nh_cities) / n_cities * 100, 1)
summary['sh_city_pct'] = round(len(sh_cities) / n_cities * 100, 1)

# NH peak months
summary['nh_peak_month_aegypti_mode'] = int(nh_cities['season_start_aegypti'].dropna().apply(
    lambda x: x).mode()[0]) if len(nh_cities) > 0 else None

# SH: % peaking Dec-Feb (austral summer)
sh_peak_aeg_dec_feb = sh_cities['peak_month_aegypti'].isin([12, 1, 2]).sum() if 'peak_month_aegypti' in sh_cities.columns else None
sh_peak_alb_dec_feb = sh_cities['peak_month_albopictus'].isin([12, 1, 2]).sum() if 'peak_month_albopictus' in sh_cities.columns else None
summary['sh_peak_dec_feb_aegypti_pct'] = round(sh_peak_aeg_dec_feb / len(sh_cities) * 100, 1) if sh_peak_aeg_dec_feb is not None else None
summary['sh_peak_dec_feb_albopictus_pct'] = round(sh_peak_alb_dec_feb / len(sh_cities) * 100, 1) if sh_peak_alb_dec_feb is not None else None

# Season length distribution — full breakdown (0–12 months) for both species
for sp in ['aegypti', 'albopictus']:
    col = f'season_{sp}'
    for months in range(0, 13):
        count = (city[col] == months).sum()
        pct = round(count / n_cities * 100, 1)
        summary[f'season_{sp}_{months}mo_count'] = int(count)
        summary[f'season_{sp}_{months}mo_pct'] = pct

# Peak month % — all cities and NH only (for tooltips)
for sp in ['aegypti', 'albopictus']:
    col = f'peak_month_{sp}'
    for m in range(1, 13):
        count_all = (city[col] == m).sum()
        count_nh = (city[city['lat'] > 0][col] == m).sum()
        n_nh = len(city[city['lat'] > 0])
        summary[f'peak_month_{sp}_{m}_count'] = int(count_all)
        summary[f'peak_month_{sp}_{m}_pct'] = round(count_all / n_cities * 100, 1)
        summary[f'peak_month_{sp}_{m}_nh_count'] = int(count_nh)
        summary[f'peak_month_{sp}_{m}_nh_pct'] = round(count_nh / n_nh * 100, 1)

print("\n=== SUMMARY STATISTICS ===")
for k, v in summary.items():
    print(f"{k}: {v}")


=== SUMMARY STATISTICS ===
total_cities: 1423
extended_suitability_aegypti: 357
extended_suitability_albopictus: 358
extended_suitability_aegypti_pct: 25.1
extended_suitability_albopictus_pct: 25.2
bimodal_aegypti: 272
bimodal_albopictus: 314
bimodal_albopictus_pct: 22.1
bimodal_non_tropical_pct: 59.9
altitude_protected: 13
altitude_protected_pct: 31.0
median_season_aegypti_tropics: 12.0
median_season_albopictus_tropics: 12.0
median_divergence_tropics: 0.0
city_count_tropics: 446
median_season_aegypti_subtropics: 5.0
median_season_albopictus_subtropics: 4.0
median_divergence_subtropics: -1.0
city_count_subtropics: 736
median_season_aegypti_temperate: 3.0
median_season_albopictus_temperate: 3.0
median_divergence_temperate: 1.0
city_count_temperate: 241
lat_gradient_aegypti_per_10deg: -2.27
lat_gradient_albopictus_per_10deg: -2.26
lat_gradient_aegypti_r2: 0.647
lat_gradient_albopictus_r2: 0.632
lat_gradient_aegypti_per_10deg_posttropics: -2.06
lat_gradient_albopictus_per_10deg_posttropi

In [6]:
# Convert to DataFrame for Tableau
summary_df = pd.DataFrame([
    {'metric': k, 'value': v, 'category': k.split('_')[0]}
    for k, v in summary.items()
])

# Add formatted display values
summary_df['display_value'] = summary_df.apply(
    lambda r: f"{r['value']:.1f}%" if 'pct' in r['metric']
    else f"{r['value']:.2f}" if isinstance(r['value'], float)
    else str(int(r['value'])),
    axis=1
)

summary_df.to_csv('summary_statistics.csv', index=False)
print(f"✓ Saved: summary_statistics.csv ({len(summary_df)} metrics)")
summary_df.head(10)

✓ Saved: summary_statistics.csv (190 metrics)


,metric,value,category,display_value
0,total_cities,1423.0,total,1423.00
1,extended_suitability_aegypti,357.0,extended,357.00
2,extended_suitability_albopictus,358.0,extended,358.00
3,extended_suitability_aegypti_pct,25.1,extended,25.1%
4,extended_suitability_albopictus_pct,25.2,extended,25.2%
5,bimodal_aegypti,272.0,bimodal,272.00
6,bimodal_albopictus,314.0,bimodal,314.00
7,bimodal_albopictus_pct,22.1,bimodal,22.1%
8,bimodal_non_tropical_pct,59.9,bimodal,59.9%
9,altitude_protected,13.0,altitude,13.00


## 3. Generate Insight Annotations (B)

In [7]:
# Create insight texts for dashboard annotations
insights_list = []

# 1. Latitude Gradient
insights_list.append({
    'insight_id': 'latitude_gradient',
    'category': 'Geographic Pattern',
    'headline': 'Latitude Drives Season Length',
    'finding': f"For every 10° of latitude away from the equator, the suitable season for Ae. aegypti declines by {abs(summary['lat_gradient_aegypti_per_10deg']):.1f} months (R²={summary['lat_gradient_aegypti_r2']:.2f}). Ae. albopictus follows a similar slope until ~45°, then levels off at 2–3 months while Ae. aegypti continues toward zero.",
    'implication': 'Season length is strongly predicted by latitude alone (R²=0.65), with deviations explained by altitude and local climate factors such as summer VPD.',
    'key_stat': f"{abs(summary['lat_gradient_aegypti_per_10deg']):.1f}",
    'stat_label': 'months lost per 10° latitude'
})

# 2. Species Gap
temperate_gap = summary['median_divergence_temperate']
insights_list.append({
    'insight_id': 'species_gap',
    'category': 'Species Comparison',
    'headline': 'Albopictus Outperforms Aegypti in Temperate Zones',
    'finding': f"Ae. albopictus follows a similar slope as Ae. aegypti until ~45°, then levels off at 2–3 months while Ae. aegypti continues toward zero. Above 50°N, the gap reaches a median of 2 months (albopictus advantage).",
    'implication': 'Photoperiod adaptation allows albopictus to colonize higher latitudes, explaining its European invasion.',
    'key_stat': f"+{temperate_gap:.0f}",
    'stat_label': 'month advantage for albopictus'
})

# 3. Bimodal Paradox
insights_list.append({
    'insight_id': 'bimodal_paradox',
    'category': 'Seasonal Pattern',
    'headline': 'Summer ≠ Peak Season Everywhere',
    'finding': f"{summary['bimodal_albopictus']} cities ({summary['bimodal_albopictus_pct']}%) show VPD-driven summer suppression, where peak suitability occurs in spring or early summer rather than midsummer. {summary['bimodal_non_tropical_pct']:.0f}% of these are in Subtropics/Temperate zones.",
    'implication': 'High summer VPD (dry heat) suppresses mosquito activity. In cities with a summer suitability dip, spring and autumn months show higher climate suitability than midsummer.',
    'key_stat': f"{summary['bimodal_albopictus']}",
    'stat_label': 'cities with summer dip'
})

# 4. Altitude Shield
insights_list.append({
    'insight_id': 'altitude_shield',
    'category': 'Geographic Pattern',
    'headline': 'Altitude Protects Tropical Cities',
    'finding': f"{summary['altitude_protected']} tropical cities above 1500m show zero Ae. aegypti season ({summary['altitude_protected_pct']:.0f}% of high-altitude tropical cities).",
    'implication': 'Elevation suppresses climate suitability even at tropical latitudes. Mexico City, Bogotá and Quito show near-zero suitability despite lying within the tropics.',
    'key_stat': f"{summary['altitude_protected']}",
    'stat_label': 'altitude-protected cities'
})

# 5. Year-Round Suitability
year_round_either     = ((city['season_aegypti'] == 12) | (city['season_albopictus'] == 12)).sum()
year_round_both       = ((city['season_aegypti'] == 12) & (city['season_albopictus'] == 12)).sum()
year_round_aegypti    = (city['season_aegypti'] == 12).sum()
year_round_albopictus = (city['season_albopictus'] == 12).sum()
total = len(city)

insights_list.append({
    'insight_id': 'year_round_suitability',
    'category': 'Climate Suitability',
    'headline': 'Year-Round Suitability Concentrated in Tropics',
    'finding': f"{year_round_either} cities ({year_round_either/total*100:.1f}%) show year-round climate suitability for at least one species. {year_round_both} cities ({year_round_both/total*100:.1f}%) show year-round suitability for both species simultaneously.",
    'implication': 'Year-round suitability is almost exclusively a tropical phenomenon, consistent with the thermal requirements of both species. In temperate regions, cold winters act as a hard seasonal constraint.',
    'key_stat': f"{year_round_either/total*100:.1f}%",
    'stat_label': 'cities with year-round suitability (≥1 species)'
})

print(f"Ae. aegypti year-round:              {year_round_aegypti} cities = {year_round_aegypti/total*100:.1f}%")
print(f"Ae. albopictus year-round:           {year_round_albopictus} cities = {year_round_albopictus/total*100:.1f}%")
print(f"Year-round (at least one species):   {year_round_either} cities = {year_round_either/total*100:.1f}%")
print(f"Year-round (both species):           {year_round_both} cities = {year_round_both/total*100:.1f}%")

# KPI: Temperate city coverage by species
temperate_cities = city[city['lat_zone'] == 'Temperate']
total_temperate = len(temperate_cities)

temperate_aegypti    = (temperate_cities['season_aegypti'] > 0).sum()
temperate_albopictus = (temperate_cities['season_albopictus'] > 0).sum()
temperate_gap        = temperate_albopictus - temperate_aegypti

print(f"Temperate cities total:                {total_temperate}")
print(f"With Ae. aegypti season:               {temperate_aegypti} = {temperate_aegypti/total_temperate*100:.1f}%")
print(f"With Ae. albopictus season:            {temperate_albopictus} = {temperate_albopictus/total_temperate*100:.1f}%")
print(f"Albopictus advantage (extra cities):   +{temperate_gap} cities = +{temperate_gap/total_temperate*100:.1f}%")


# 6. Season Timing (Temperate)
insights_list.append({
    'insight_id': 'season_timing',
    'category': 'Public Health Planning',
    'headline': 'Temperate Season Window: May–October',
    'finding': f"Median albopictus season in temperate zones starts month {summary['temperate_median_start_albopictus']:.0f} and ends month {summary['temperate_median_end_albopictus']:.0f}.",
    'implication': 'In temperate zones, the climate window for mosquito activity is tightly bounded, roughly May through August, with cold winters acting as a hard seasonal constraint.',
    'key_stat': f"{summary['temperate_median_start_albopictus']:.0f}–{summary['temperate_median_end_albopictus']:.0f}",
    'stat_label': 'season months (temperate)'
})

insights_df = pd.DataFrame(insights_list)
insights_df.to_csv('insight_annotations.csv', index=False)
print(f"Saved: insight_annotations.csv ({len(insights_df)} insights)")
insights_df[['insight_id', 'headline', 'key_stat', 'stat_label']]

Ae. aegypti year-round:              268 cities = 18.8%
Ae. albopictus year-round:           301 cities = 21.2%
Year-round (at least one species):   312 cities = 21.9%
Year-round (both species):           257 cities = 18.1%
Temperate cities total:                241
With Ae. aegypti season:               197 = 81.7%
With Ae. albopictus season:            240 = 99.6%
Albopictus advantage (extra cities):   +43 cities = +17.8%
Saved: insight_annotations.csv (6 insights)


,insight_id,headline,key_stat,stat_label
0,latitude_gradient,Latitude Drives Season Length,2.3,months lost per 10° latitude
1,species_gap,Albopictus Outperforms Aegypti in Temperate Zones,+1,month advantage for albopictus
2,bimodal_paradox,Summer ≠ Peak Season Everywhere,314,cities with summer dip
3,altitude_shield,Altitude Protects Tropical Cities,13,altitude-protected cities
4,year_round_suitability,Year-Round Suitability Concentrated in Tropics,21.9%,cities with year-round suitability (≥1 species)
5,season_timing,Temperate Season Window: May–October,6–8,season months (temperate)


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Print formatted insights for README/documentation

print("DASHBOARD INSIGHTS")


for i, row in insights_df.iterrows():
    print(f"\n### {i+1}. {row['headline']}")
    print(f"**Finding:** {row['finding']}")
    print(f"**So what:** {row['implication']}")
    print(f"**Key stat:** {row['key_stat']} {row['stat_label']}")

DASHBOARD INSIGHTS

### 1. Latitude Drives Season Length
**Finding:** For every 10° of latitude away from the equator, the suitable season for Ae. aegypti declines by 2.3 months (R²=0.65). Ae. albopictus follows a similar slope until ~45°, then levels off at 2–3 months while Ae. aegypti continues toward zero.
**So what:** Season length is strongly predicted by latitude alone (R²=0.65), with deviations explained by altitude and local climate factors such as summer VPD.
**Key stat:** 2.3 months lost per 10° latitude

### 2. Albopictus Outperforms Aegypti in Temperate Zones
**Finding:** Ae. albopictus follows a similar slope as Ae. aegypti until ~45°, then levels off at 2–3 months while Ae. aegypti continues toward zero. Above 50°N, the gap reaches a median of 2 months (albopictus advantage).
**So what:** Photoperiod adaptation allows albopictus to colonize higher latitudes, explaining its European invasion.
**Key stat:** +1 month advantage for albopictus

### 3. Summer ≠ Peak Season Ever

## 4. Extend City Insights with Story Flags (C)

In [10]:
# Add analytical flags for Tableau filtering/highlighting

# Flag 1: Temperate Albopictus Advantage
# Cities where albopictus thrives but aegypti struggles
city['temperate_albopictus_advantage'] = (
    (city['lat_zone'] == 'Temperate') &
    (city['species_divergence'] > 0) &
    (city['season_albopictus'] >= 4)
)

# Flag 2: Tropical Hotspot
# High-suitability tropical cities — year-round climate suitability
city['tropical_hotspot'] = (
    (city['lat_zone'] == 'Tropics') &
    (city['suitability_tier_aegypti'] == 'Extended')
)

# Flag 3: Invasion Front
# Temperate cities where albopictus shows extended climate suitability (≥4 months)
city['invasion_front'] = (
    (city['lat_zone'] == 'Temperate') &
    (city['season_albopictus'] >= 4) &
    (city['suitability_tier_albopictus'].isin(['Seasonal', 'Extended']))
)

# Flag 4: Mediterranean Pattern
# Bimodal cities outside tropics — VPD-driven summer dip
city['mediterranean_pattern'] = (
    (city['bimodal_flag_albopictus']) &
    (city['lat_zone'].isin(['Subtropics', 'Temperate']))
)

# Flag 5: Year-Round Suitability
# Cities with 12-month season for either species
city['year_round_suitability'] = (
    (city['season_aegypti'] == 12) |
    (city['season_albopictus'] == 12)
)

# Flag 6: Emerging Suitability (Subtropical)
# Subtropical cities with seasonal or extended climate suitability — expansion zone
city['emerging_suitability_zone'] = (
    (city['lat_zone'] == 'Subtropics') &
    (city['suitability_tier_albopictus'].isin(['Seasonal', 'Extended']))
)


print("\n=== FLAG SUMMARY ===")
flag_cols = ['temperate_albopictus_advantage', 'tropical_hotspot', 'invasion_front',
             'mediterranean_pattern', 'year_round_suitability', 'emerging_suitability_zone', 'altitude_suppressed']
for flag in flag_cols:
    count = city[flag].sum()
    pct = count / len(city) * 100
    print(f"{flag}: {count} cities ({pct:.1f}%)")


=== FLAG SUMMARY ===
temperate_albopictus_advantage: 58 cities (4.1%)
tropical_hotspot: 326 cities (22.9%)
invasion_front: 104 cities (7.3%)
mediterranean_pattern: 188 cities (13.2%)
year_round_suitability: 312 cities (21.9%)
emerging_suitability_zone: 582 cities (40.9%)
altitude_suppressed: 13 cities (0.9%)


In [11]:
# Create story_tag column for easy filtering
def assign_story_tag(row):
    """Assign primary story tag (hierarchical priority)."""
    if row['altitude_suppressed']:
        return 'Altitude Protected'
    elif row['tropical_hotspot']:
        return 'Tropical Hotspot'
    elif row['mediterranean_pattern']:
        return 'Mediterranean Pattern'
    elif row['invasion_front']:
        return 'Invasion Front'
    elif row['year_round_suitability']:
        return 'Year-Round Suitability'
    elif row['emerging_suitability_zone']:
        return 'Emerging Suitability'
    elif row['temperate_albopictus_advantage']:
        return 'Albopictus Advantage'
    else:
        return 'Standard'

city['story_tag'] = city.apply(assign_story_tag, axis=1)

print("\n=== STORY TAG DISTRIBUTION ===")
print(city['story_tag'].value_counts())


=== STORY TAG DISTRIBUTION ===
story_tag
Emerging Suitability      405
Standard                  352
Tropical Hotspot          326
Mediterranean Pattern     188
Invasion Front             93
Year-Round Suitability     46
Altitude Protected         13
Name: count, dtype: int64


In [12]:
# Round numeric columns
round_cols = ['cumulative_aegypti', 'cumulative_albopictus', 'peak_score_aegypti',
              'peak_score_albopictus', 'summer_vpd']
for col in round_cols:
    city[col] = city[col].round(3)

# Reorder columns
column_order = [
    # Identifiers
    'city_key', 'city', 'country', 'lat', 'lon', 'abs_lat', 'population', 'elevation_m',
    # Classification
    'lat_zone', 'story_tag',
    # Ae. aegypti
    'season_aegypti', 'season_start_aegypti', 'season_end_aegypti',
    'cumulative_aegypti', 'peak_score_aegypti',
    'bimodal_flag_aegypti', 'suitability_tier_aegypti',
    # Ae. albopictus
    'season_albopictus', 'season_start_albopictus', 'season_end_albopictus',
    'cumulative_albopictus', 'peak_score_albopictus',
    'bimodal_flag_albopictus', 'suitability_tier_albopictus',
    # Comparative
    'species_divergence', 'summer_vpd',
    # Story flags
    'altitude_suppressed', 'tropical_hotspot', 'invasion_front',
    'mediterranean_pattern', 'year_round_suitability', 'emerging_suitability_zone',
    'temperate_albopictus_advantage'
]


city = city[column_order]
city.to_csv('city_insights_extended.csv', index=False)
print(f"✓ Saved: city_insights_extended.csv ({len(city)} cities × {len(city.columns)} columns)")

✓ Saved: city_insights_extended.csv (1423 cities × 33 columns)


In [14]:
# Sample cities for each story tag
print("\n SAMPLE CITIES BY STORY TAG ")
for tag in city['story_tag'].unique():
    sample = city[city['story_tag'] == tag].head(3)
    cities = ', '.join(sample['city'].tolist())
    print(f"\n{tag}:")
    print(f"  {cities}")


 SAMPLE CITIES BY STORY TAG 

Emerging Suitability:
  Tokyo, Shanghai, São Paulo

Tropical Hotspot:
  Jakarta, Guangzhou, Mumbai

Standard:
  Delhi, Moscow, Tehran

Altitude Protected:
  Mexico City, Bogotá, Addis Ababa

Year-Round Suitability:
  Karachi, Lima, Miami

Invasion Front:
  New York, Istanbul, Chicago

Mediterranean Pattern:
  Xi’an, Wuhan, Hangzhou


In [15]:
print("OUTPUTS GENERATED")

print("\n1. summary_statistics.csv")
print("- KPI boxes, BANs")
print(f"- {len(summary_df)} metrics")

print("\n2. insight_annotations.csv")
print("- Dashboard text annotations")
print(f"- {len(insights_df)} insights")

print("\n3. city_insights_extended.csv")
print("- Enhanced city data with story flags")
print(f"- {len(city)} cities × {len(city.columns)} columns")
print(f"- Includes: story_tag + 7 boolean flags")

OUTPUTS GENERATED

1. summary_statistics.csv
- KPI boxes, BANs
- 190 metrics

2. insight_annotations.csv
- Dashboard text annotations
- 6 insights

3. city_insights_extended.csv
- Enhanced city data with story flags
- 1423 cities × 33 columns
- Includes: story_tag + 7 boolean flags
